# FarmFederate â€” Real Data Training Pipeline

Trains **24 models + RAG pipeline** using real local data:
- **Text**: per-class `text.csv` (cleaned BLIP captions + pvvqa)
- **Images**: per-class `images/` folders (PlantVillage + Beans)

### Part A â€” Classification Models (24 total)
| Type | Models | Count |
|------|--------|-------|
| LLM  | DistilBERT, BERT-tiny, RoBERTa-tiny, ALBERT-tiny, MobileBERT | 5 |
| ViT  | ViT-Base, DeiT-tiny, Swin-tiny, ConvNeXT-tiny, EfficientNet | 5 |
| VLM  | concat, attention, gated, clip, flamingo, blip2, coca, unified_io | 8 |
| Fed/Cent | LLM, ViT, VLM (centralized + federated) | 6 |

### Part B â€” RAG Pipeline
| Step | What |
|------|------|
| Encoder | DistilRoBERTa text + CNN vision â†’ fused 512-d |
| Federated training | FedAvg across farms, InfoNCE + advisory BCE loss |
| Knowledge base | Per-farm FAISS index (never transmitted to server) |
| RAG | classify â†’ query_builder â†’ FAISS retrieve â†’ context assemble |
| Metrics | Recall@5, MRR, NDCG@5, embedding drift |

### Setup
1. **Runtime â†’ Change runtime type â†’ T4 GPU**
2. Upload your `data/` folder to Google Drive at `MyDrive/FarmFederate/data/`
3. Run all cells

In [ ]:
# â”€â”€ Cell 1: Mount Google Drive â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# â”€â”€ Cell 2: Clone repo â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os

REPO_URL = 'https://github.com/Solventerritory/FarmFederate-Advisor.git'
REPO_DIR = '/content/FarmFederate'
BRANCH   = 'feature/multimodal-work'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print('Repo ready at', REPO_DIR)

In [ ]:
# â”€â”€ Cell 3: Copy data from Drive â†’ Colab â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import shutil, os

DRIVE_DATA = '/content/drive/MyDrive/FarmFederate/data'
LOCAL_DATA = '/content/FarmFederate/data'

if os.path.exists(DRIVE_DATA):
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print(f'Data copied: {DRIVE_DATA} -> {LOCAL_DATA}')
else:
    print(f'WARNING: {DRIVE_DATA} not found.')
    print('Upload your data/ folder to Drive at MyDrive/FarmFederate/data/')

# Verify
print('\nData summary:')
print(f'{"Class":<15} {"Texts":>8} {"Images":>8}')
print('-' * 33)
for cls in ['water_stress', 'nutrient_def', 'pest_risk', 'disease_risk', 'heat_stress']:
    txt_path = f'{LOCAL_DATA}/{cls}/text.csv'
    img_path = f'{LOCAL_DATA}/{cls}/images'
    txt  = len(open(txt_path).readlines()) - 1 if os.path.exists(txt_path) else 0
    imgs = len([f for f in os.listdir(img_path) if f.endswith(('.jpg', '.png'))]) if os.path.exists(img_path) else 0
    print(f'{cls:<15} {txt:>8} {imgs:>8}')

In [ ]:
# â”€â”€ Cell 4: Install dependencies â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
!pip install -q torch torchvision transformers scikit-learn pandas numpy \
               matplotlib seaborn tqdm pillow faiss-cpu datasets
print('Dependencies installed.')

In [ ]:
# â”€â”€ Cell 5: GPU check â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU â€” Runtime â†’ Change runtime type â†’ T4 GPU')

---
## Part A â€” Classification Training (LLM + ViT + VLM)

In [ ]:
# â”€â”€ Cell 6: Configuration â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import sys
sys.path.insert(0, '/content/FarmFederate/backend')

from FarmFederate_Colab_Complete import Config
from pathlib import Path

config = Config(
    epochs                = 15,
    batch_size            = 16,
    max_samples_per_class = 800,
    fed_rounds            = 5,
    num_clients           = 3,
    learning_rate         = 1e-4,
)

# Save to Drive â€” survives Colab disconnects
config.checkpoint_dir = Path('/content/drive/MyDrive/FarmFederate/checkpoints')
config.output_dir     = Path('/content/drive/MyDrive/FarmFederate/outputs')
config.plots_dir      = Path('/content/drive/MyDrive/FarmFederate/plots')

for d in [config.checkpoint_dir, config.output_dir, config.plots_dir]:
    d.mkdir(parents=True, exist_ok=True)

print('Configuration:')
for k in ['epochs', 'batch_size', 'max_samples_per_class', 'fed_rounds', 'num_clients', 'learning_rate']:
    print(f'  {k:<25}: {getattr(config, k)}')
print(f'  {"checkpoint_dir":<25}: {config.checkpoint_dir}')

In [ ]:
# â”€â”€ Cell 7: Run classification training (24 models) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 5 LLM + 5 ViT + 8 VLM + 3 centralized + 3 federated
# Checkpoints auto-saved to Drive after each model

from FarmFederate_Colab_Complete import run_training_real_data

results = run_training_real_data(config=config)

In [ ]:
# â”€â”€ Cell 8: Results summary table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('=' * 70)
print('CLASSIFICATION RESULTS SUMMARY')
print('=' * 70)

print(f'\n{"Model":<22} {"F1 Micro":>10} {"F1 Macro":>10} {"Accuracy":>10} {"Params":>12}')
print('-' * 68)

print('\n--- LLM Models (Text) ---')
for name, r in results['llm_models'].items():
    print(f'{name:<22} {r["f1"]:>10.4f} {r["f1_macro"]:>10.4f} {r["accuracy"]:>10.4f} {r["params"]:>12,}')

print('\n--- ViT Models (Image) ---')
for name, r in results['vit_models'].items():
    print(f'{name:<22} {r["f1"]:>10.4f} {r["f1_macro"]:>10.4f} {r["accuracy"]:>10.4f} {r["params"]:>12,}')

print('\n--- VLM Fusion Models (Text + Image) ---')
for name, r in results['vlm_models'].items():
    print(f'{name:<22} {r["f1"]:>10.4f} {r["f1_macro"]:>10.4f} {r["accuracy"]:>10.4f} {r["params"]:>12,}')

print('\n--- Centralized vs Federated ---')
print(f'{"Type":<10} {"Centralized F1":>16} {"Federated F1":>14} {"Diff":>8} {"Winner":>12}')
print('-' * 64)
for mt in ['LLM', 'ViT', 'VLM']:
    c = results['centralized'][mt]['f1']
    f = results['federated'][mt]['f1']
    d = f - c
    w = 'Federated' if d > 0 else ('Centralized' if d < 0 else 'Tie')
    print(f'{mt:<10} {c:>16.4f} {f:>14.4f} {d:>+8.4f} {w:>12}')

all_f1 = {}
for n, r in results['llm_models'].items(): all_f1[f'LLM-{n}'] = r['f1']
for n, r in results['vit_models'].items(): all_f1[f'ViT-{n}'] = r['f1']
for n, r in results['vlm_models'].items(): all_f1[f'VLM-{n}'] = r['f1']
best = max(all_f1, key=all_f1.get)
print(f'\nBest model: {best}  F1={all_f1[best]:.4f}')

In [ ]:
# â”€â”€ Cell 9: Training curves â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, title, group in zip(axes,
                             ['LLM Models', 'ViT Models', 'VLM Fusion'],
                             [results['llm_models'], results['vit_models'], results['vlm_models']]):
    for name, r in group.items():
        val_f1 = r.get('history', {}).get('val_f1', [])
        if val_f1:
            ax.plot(val_f1, label=name, marker='o', markersize=3)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Val F1')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FarmFederate/plots/training_curves.png', dpi=150)
plt.show()
print('Saved: training_curves.png')

In [ ]:
# â”€â”€ Cell 10: F1 bar chart â€” all models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib.pyplot as plt, numpy as np
from matplotlib.patches import Patch

names, f1s, colors = [], [], []
for n, r in results['llm_models'].items(): names.append(n); f1s.append(r['f1']); colors.append('#2196F3')
for n, r in results['vit_models'].items(): names.append(n); f1s.append(r['f1']); colors.append('#4CAF50')
for n, r in results['vlm_models'].items(): names.append(n); f1s.append(r['f1']); colors.append('#FF5722')

fig, ax = plt.subplots(figsize=(16, 6))
bars = ax.bar(range(len(names)), f1s, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('F1 Score (Micro)')
ax.set_title('FarmFederate â€” All Models F1 Comparison (Real Data)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.0)
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}',
            ha='center', va='bottom', fontsize=7)
ax.legend(handles=[
    Patch(color='#2196F3', label='LLM (Text)'),
    Patch(color='#4CAF50', label='ViT (Image)'),
    Patch(color='#FF5722', label='VLM (Multimodal)'),
    plt.Line2D([0],[0], color='black', linestyle='--', label=f'Mean={np.mean(f1s):.3f}'),
])
ax.axhline(y=np.mean(f1s), color='black', linestyle='--', alpha=0.4)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FarmFederate/plots/f1_comparison_all_models.png', dpi=150)
plt.show()
print('Saved: f1_comparison_all_models.png')

In [ ]:
# â”€â”€ Cell 11: Centralized vs Federated chart â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib.pyplot as plt, numpy as np

model_types = ['LLM', 'ViT', 'VLM']
cent_f1s = [results['centralized'][mt]['f1'] for mt in model_types]
fed_f1s  = [results['federated'][mt]['f1']   for mt in model_types]

x, w = np.arange(len(model_types)), 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, cent_f1s, w, label='Centralized', color='#1976D2')
b2 = ax.bar(x + w/2, fed_f1s,  w, label='Federated',   color='#388E3C')
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(model_types, fontsize=12)
ax.set_ylabel('F1 Score (Micro)')
ax.set_title('Centralized vs Federated Learning', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.0); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FarmFederate/plots/centralized_vs_federated.png', dpi=150)
plt.show()
print('Saved: centralized_vs_federated.png')

In [ ]:
# ── Cell 12: Per-class F1 heatmap — all models ───────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

STRESS_CLASSES = ['water_stress', 'nutrient_def', 'pest_risk', 'disease_risk', 'heat_stress']

all_names, all_perclass = [], []
for mtype, group in [('LLM', results['llm_models']),
                     ('ViT', results['vit_models']),
                     ('VLM', results['vlm_models'])]:
    for name, r in group.items():
        pc = r.get('per_class_f1', r.get('per_class', None))
        if pc is not None:
            all_names.append(f'{mtype}-{name}')
            all_perclass.append(list(pc.values()) if isinstance(pc, dict) else list(pc))

if all_perclass:
    matrix = np.array(all_perclass)
    short_cls = [c.replace('_', '
') for c in STRESS_CLASSES]
    fig, ax = plt.subplots(figsize=(10, max(5, len(all_names) * 0.45)))
    sns.heatmap(matrix, annot=True, fmt='.2f', cmap='RdYlGn',
                xticklabels=short_cls, yticklabels=all_names,
                vmin=0, vmax=1, linewidths=0.4, ax=ax,
                cbar_kws={'label': 'F1 Score'})
    ax.set_title('Per-Class F1 Score — All Models', fontsize=13, fontweight='bold')
    ax.set_xlabel('Stress Class'); ax.set_ylabel('Model')
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/FarmFederate/plots/per_class_f1_heatmap.png', dpi=150)
    plt.show()
    print('Saved: per_class_f1_heatmap.png')
else:
    print('per_class_f1 not in results — skipping heatmap')

In [ ]:
# ── Cell 13: Model complexity vs accuracy scatter ────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(10, 6))
_colors  = {'LLM': '#2196F3', 'ViT': '#4CAF50', 'VLM': '#FF5722'}
_markers = {'LLM': 'o',       'ViT': 's',        'VLM': '^'}

for mtype, group in [('LLM', results['llm_models']),
                     ('ViT', results['vit_models']),
                     ('VLM', results['vlm_models'])]:
    for name, r in group.items():
        params = r.get('params', 0)
        f1     = r.get('f1', 0)
        if params > 0:
            ax.scatter(params / 1e6, f1,
                       color=_colors[mtype], marker=_markers[mtype],
                       s=110, edgecolors='white', linewidths=0.6, zorder=3)
            ax.annotate(name, (params / 1e6, f1),
                        textcoords='offset points', xytext=(5, 3),
                        fontsize=7, color=_colors[mtype])

ax.legend(handles=[Patch(color=c, label=f'{t} Models') for t, c in _colors.items()], fontsize=10)
ax.set_xlabel('Model Parameters (Millions)', fontsize=11)
ax.set_ylabel('F1 Score (Micro)', fontsize=11)
ax.set_title('Model Complexity vs Classification Accuracy', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FarmFederate/plots/complexity_vs_accuracy.png', dpi=150)
plt.show()
print('Saved: complexity_vs_accuracy.png')

In [ ]:
# â”€â”€ Cell 12: Save classification results â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import json, shutil

results_path = '/content/drive/MyDrive/FarmFederate/outputs/complete_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'Results saved: {results_path}')

zip_path = '/content/drive/MyDrive/FarmFederate/farmfederate_checkpoints'
shutil.make_archive(zip_path, 'zip', str(config.checkpoint_dir))
print(f'Checkpoints zipped: {zip_path}.zip')

---
## Part B â€” RAG Pipeline (Federated Retrieval-Augmented Generation)

Trains a federated RAG system on top of the same real data:
- **Encoder**: DistilRoBERTa (text) + small CNN (image) â†’ fused 512-d representation
- **Knowledge base**: Per-farm FAISS index built from agronomic seed documents â€” never transmitted to server
- **Federated training**: FedAvg with InfoNCE contrastive loss (retriever) + BCE advisory loss
- **Evaluation**: Recall@5, MRR, NDCG@5, per-class F1, embedding drift per round

Execution modes (set in Cell 13):
| Mode | Farms | FL rounds | Samples/farm | Time |
|------|-------|-----------|--------------|------|
| `quick` | 2 | 2 | 100 | ~5 min |
| `standard` | 3 | 5 | 400 | ~20 min |
| `full` | 5 | 8 | 800 | ~40 min |

In [ ]:
# â”€â”€ Cell 13: RAG configuration â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os

# â”€â”€ Set mode: 'quick' | 'standard' | 'full' â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
RAG_MODE       = 'standard'   # change this
GOOGLE_API_KEY = ''           # optional: set for Gemini advisory generation

# Output dirs on Drive
RAG_OUT_DIR  = '/content/drive/MyDrive/FarmFederate/rag_results'
RAG_CKPT_DIR = '/content/drive/MyDrive/FarmFederate/rag_checkpoints'
os.makedirs(RAG_OUT_DIR,  exist_ok=True)
os.makedirs(RAG_CKPT_DIR, exist_ok=True)

_MODES = {
    'quick':    dict(num_farms=2, fed_rounds=2,  local_epochs=1, samples_per_farm=100, rag_rounds=2,  top_k=3),
    'standard': dict(num_farms=3, fed_rounds=5,  local_epochs=2, samples_per_farm=400, rag_rounds=10, top_k=5),
    'full':     dict(num_farms=5, fed_rounds=8,  local_epochs=3, samples_per_farm=800, rag_rounds=15, top_k=5),
}
cfg = _MODES[RAG_MODE]
print(f'RAG mode : {RAG_MODE}')
print(f'Farms    : {cfg["num_farms"]}')
print(f'FL rounds: {cfg["fed_rounds"]}')
print(f'Samples/farm: {cfg["samples_per_farm"]}')
print(f'RAG rounds  : {cfg["rag_rounds"]}')
print(f'Top-K       : {cfg["top_k"]}')
print(f'Output dir  : {RAG_OUT_DIR}')

In [ ]:
# â”€â”€ Cell 14: Run RAG pipeline â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Patches EXECUTION_MODE and output dirs before importing the RAG script,
# then calls run() which handles everything end-to-end:
#   data load â†’ multimodal training â†’ KB build â†’ federated RAG training â†’ eval â†’ plots

import sys, importlib
sys.path.insert(0, '/content/FarmFederate/backend')

# Inject config into module namespace before any module-level code runs
import builtins
_orig_open = builtins.open

# Set globals that the RAG script reads at import time
import types
rag_mod = types.ModuleType('FarmFederate_RAG_Colab')
rag_mod.__file__ = '/content/FarmFederate/backend/FarmFederate_RAG_Colab.py'

# Read source, inject EXECUTION_MODE and output dirs before exec
with open('/content/FarmFederate/backend/FarmFederate_RAG_Colab.py', 'r') as _f:
    _src = _f.read()

# Override the mode and key at the top
_src = _src.replace(
    'EXECUTION_MODE = "standard"',
    f'EXECUTION_MODE = "{RAG_MODE}"'
).replace(
    'GOOGLE_API_KEY = ""',
    f'GOOGLE_API_KEY = "{GOOGLE_API_KEY}"'
).replace(
    'OUT_DIR = Path("rag_results"); OUT_DIR.mkdir(exist_ok=True)',
    f'OUT_DIR = Path("{RAG_OUT_DIR}"); OUT_DIR.mkdir(parents=True, exist_ok=True)'
).replace(
    'CKPT_DIR = Path("rag_checkpoints"); CKPT_DIR.mkdir(exist_ok=True)',
    f'CKPT_DIR = Path("{RAG_CKPT_DIR}"); CKPT_DIR.mkdir(parents=True, exist_ok=True)'
)

sys.modules['FarmFederate_RAG_Colab'] = rag_mod
exec(compile(_src, '/content/FarmFederate/backend/FarmFederate_RAG_Colab.py', 'exec'),
     rag_mod.__dict__)

# Run the full pipeline
rag_results = rag_mod.run()
print('\nRAG pipeline complete.')

In [ ]:
# â”€â”€ Cell 15: RAG metrics summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import json

print('=' * 60)
print('RAG PIPELINE â€” RESULTS SUMMARY')
print('=' * 60)

# rag_results keys depend on run() return value; handle both dict and None
if rag_results and isinstance(rag_results, dict):
    ret = rag_results.get('retrieval_metrics', rag_results.get('ret_metrics', {}))
    cls = rag_results.get('classification', rag_results.get('cls_metrics', {}))
    top_k = cfg['top_k']

    print(f'\n--- Retrieval Metrics ---')
    for key in [f'recall_at_{top_k}', 'mrr', f'ndcg_at_{top_k}', 'kb_coverage']:
        val = ret.get(key, ret.get(key.replace(f'_{top_k}', '_5'), 'N/A'))
        label = key.replace('_', ' ').replace('at', '@').upper()
        print(f'  {label:<20}: {val:.4f}' if isinstance(val, float) else f'  {label:<20}: {val}')

    print(f'\n--- Classification Metrics ---')
    for key in ['f1_macro', 'f1_micro', 'accuracy']:
        val = cls.get(key, 'N/A')
        print(f'  {key:<20}: {val:.4f}' if isinstance(val, float) else f'  {key:<20}: {val}')

    print(f'\n--- Federated Training ---')
    fed = rag_results.get('federated', {})
    print(f'  Rounds completed : {fed.get("rounds", cfg["fed_rounds"])}')
    print(f'  Final retriever loss: {fed.get("final_ret_loss", "N/A")}')
    print(f'  Embedding drift  : {fed.get("drift", rag_results.get("drift", "N/A"))}')

    # Save full RAG results
    rag_json_path = f'{RAG_OUT_DIR}/rag_results_summary.json'
    with open(rag_json_path, 'w') as f:
        json.dump(rag_results, f, indent=2, default=str)
    print(f'\nFull results saved: {rag_json_path}')
else:
    print('No structured results returned â€” check plots in', RAG_OUT_DIR)

In [ ]:
# â”€â”€ Cell 16: Display RAG plots â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plot_files = sorted([
    f for f in os.listdir(RAG_OUT_DIR)
    if f.endswith('.png')
])

print(f'Found {len(plot_files)} RAG plots in {RAG_OUT_DIR}')

if plot_files:
    n_cols = 2
    n_rows = (len(plot_files) + 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for ax, fname in zip(axes, plot_files):
        img = mpimg.imread(os.path.join(RAG_OUT_DIR, fname))
        ax.imshow(img)
        ax.set_title(fname.replace('.png', '').replace('_', ' '), fontsize=10)
        ax.axis('off')

    # Hide unused axes
    for ax in axes[len(plot_files):]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'{RAG_OUT_DIR}/rag_all_plots.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved combined plot: {RAG_OUT_DIR}/rag_all_plots.png')

In [ ]:
# â”€â”€ Cell 17: Combined summary â€” Classification + RAG â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import numpy as np

print('=' * 70)
print('FARMFEDERATE â€” FULL PIPELINE SUMMARY')
print('=' * 70)

# Best classification model
all_f1 = {}
for n, r in results['llm_models'].items(): all_f1[f'LLM-{n}'] = r['f1']
for n, r in results['vit_models'].items(): all_f1[f'ViT-{n}'] = r['f1']
for n, r in results['vlm_models'].items(): all_f1[f'VLM-{n}'] = r['f1']
best_cls = max(all_f1, key=all_f1.get)

print(f'\nClassification (24 models trained on real data):')
print(f'  Best model   : {best_cls}  F1={all_f1[best_cls]:.4f}')
print(f'  LLM avg F1   : {np.mean([r["f1"] for r in results["llm_models"].values()]):.4f}')
print(f'  ViT avg F1   : {np.mean([r["f1"] for r in results["vit_models"].values()]):.4f}')
print(f'  VLM avg F1   : {np.mean([r["f1"] for r in results["vlm_models"].values()]):.4f}')

print(f'\nFederated vs Centralized:')
for mt in ['LLM', 'ViT', 'VLM']:
    c = results['centralized'][mt]['f1']
    f = results['federated'][mt]['f1']
    print(f'  {mt}: Centralized={c:.4f}  Federated={f:.4f}  ({f-c:+.4f})')

print(f'\nRAG Pipeline ({RAG_MODE} mode, {cfg["num_farms"]} farms, {cfg["fed_rounds"]} rounds):')
if rag_results and isinstance(rag_results, dict):
    ret = rag_results.get('retrieval_metrics', rag_results.get('ret_metrics', {}))
    top_k = cfg['top_k']
    print(f'  Recall@{top_k}     : {ret.get(f"recall_at_{top_k}", "N/A")}')
    print(f'  MRR          : {ret.get("mrr", "N/A")}')
    print(f'  NDCG@{top_k}       : {ret.get(f"ndcg_at_{top_k}", "N/A")}')
    print(f'  KB Coverage  : {ret.get("kb_coverage", "N/A")}')
else:
    print('  See RAG results JSON for metrics.')

print(f'\nAll outputs saved to Google Drive:')
print(f'  MyDrive/FarmFederate/outputs/          (classification results)')
print(f'  MyDrive/FarmFederate/plots/            (classification plots)')
print(f'  MyDrive/FarmFederate/checkpoints/      (model checkpoints)')
print(f'  MyDrive/FarmFederate/rag_results/      (RAG plots + metrics)')
print(f'  MyDrive/FarmFederate/rag_checkpoints/  (RAG model checkpoint)')

In [ ]:
# â”€â”€ Cell 18: Zip everything and download â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import shutil
from google.colab import files

# Zip classification checkpoints
shutil.make_archive(
    '/content/drive/MyDrive/FarmFederate/farmfederate_checkpoints',
    'zip', str(config.checkpoint_dir)
)

# Zip RAG checkpoints
shutil.make_archive(
    '/content/drive/MyDrive/FarmFederate/rag_checkpoints_archive',
    'zip', RAG_CKPT_DIR
)

print('All zipped to Drive.')
print('  farmfederate_checkpoints.zip  â€” classification models')
print('  rag_checkpoints_archive.zip   â€” RAG model')

# Optional: download directly to your machine
# files.download('/content/drive/MyDrive/FarmFederate/farmfederate_checkpoints.zip')
# files.download('/content/drive/MyDrive/FarmFederate/rag_checkpoints_archive.zip')